# 2. 최종 모델: 0.709 앙상블 + TabICL (리더보드 0.711)

- 최종 확률 = (랜덤 포레스트 + Extra Trees + 스태킹) / 3 + 0.25 × TabICL
- 테스트 중 학습 데이터와 성분이 같은 303행은 학습 데이터의 품질을 그대로 쓴다.
- 리더보드 정확도 0.711(0.709 모델보다 2행 더 맞힘), 그룹 교차검증에서 0.709 모델보다 +0.0042(분할 3가지 모두 상승)이다.

이 노트북은 코드 셀마다 "왜 이 단계가 필요한가"를 앞에 적었다. 전체 흐름은 다음 인과 관계로 이어진다.

1. 학습 데이터에 같은 성분의 와인이 반복되고 테스트에도 303행이 그대로 나온다 → 그룹 교차검증으로 모델을 고르고, 303행은 학습 데이터 품질로 맞춘다.
2. 품질과 성분의 관계가 비선형이고 인접 등급이 크게 겹친다 → 여러 성분 조합을 학습하는 트리 앙상블(랜덤 포레스트, Extra Trees, CatBoost)과 이를 결합하는 스태킹으로 0.709 모델을 만든다.
3. 0.709 이후 트리 계열 변형 약 100가지가 리더보드에서 모두 0.709를 넘지 못했고, 원인은 트리 모델들이 같은 행에서 함께 틀리기 때문이었다 → 트리와 원리가 다른 사전학습 모델 TabICL을 더해 서로 다른 오답을 보완한다.
4. TabICL을 0.25 비중으로 섞으면 0.709의 판단은 대부분 유지되고 확신이 낮은 경계 행만 바뀐다 → 리더보드 0.711.

## 2-1. 데이터 불러오기와 전처리

**왜 필요한가**
- 팀 공통 코드 `common.py`로 데이터를 불러오고 전처리한다. 모든 팀원이 같은 입력을 써야 모델 결과를 서로 비교할 수 있기 때문이다.
- `make_xy`는 [01_eda_preprocessing.ipynb](01_eda_preprocessing.ipynb)의 결론대로 `index`를 제거하고, 종류를 숫자로 바꾸며, 결합 이산화황(`bound_so2` = 총 이산화황 − 독립 이산화황)을 추가한다.
- 랜덤 포레스트·Extra Trees에는 `bound_so2`를 빼고 MinMax 스케일링한 입력을 쓴다. 이 두 가지는 공개 코드를 재현하면서 리더보드가 오른(0.699 → 0.705 → 0.708) 설정이며, 그룹 교차검증에서는 차이가 없었다. 따라서 성능 근거라기보다 0.709 모델을 그대로 재현하기 위해 유지한다.
- 스태킹은 팀 공통 전처리 입력(`bound_so2` 포함)을 그대로 쓴다.

In [1]:
from common import *
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from tabicl import TabICLClassifier


train, test, sub = load_data()
X_common, y = make_xy(train)
X_test_common = make_x(test)

scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(X_common.drop(columns='bound_so2')), columns=X_common.columns.drop('bound_so2'))
X_test = pd.DataFrame(scaler.transform(X_test_common.drop(columns='bound_so2')), columns=X_common.columns.drop('bound_so2'))

/Users/gimhuigyeong/Downloads/wine_0711_mac/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2-2. 0.709 모델: 랜덤 포레스트 + Extra Trees + 스태킹

- 모델 구조와 하이퍼파라미터 출처: [25-2 YB 1조 미니 프로젝트 Method 1](https://github.com/gravity102424/ESAA/blob/main/ESAA_YB_MiniProject(2).ipynb)

**왜 트리 앙상블인가**
- EDA에서 등급별 성분 분포가 크게 겹쳐, 한 변수의 기준선으로는 등급을 나눌 수 없었다. 여러 성분의 조합으로 경계를 나누는 트리 모델이 이 구조에 맞는다.
- 단일 결정 트리는 학습 데이터에 과적합하므로, 여러 트리를 평균하는 랜덤 포레스트(부트스트랩 표본)와 Extra Trees(무작위 분할점)를 함께 쓴다. 두 모델은 트리를 만드는 방식이 달라 서로의 오차를 일부 상쇄한다.

**왜 스태킹을 더하는가**
- 랜덤 포레스트·Extra Trees·CatBoost의 등급별 확률을 로지스틱 회귀로 결합하면, 모델마다 잘 맞히는 등급의 차이를 학습해 단순 평균보다 나은 결합 비율을 얻는다.
- 메타 모델의 학습 확률은 성분이 같은 와인을 한 폴드로 묶은 3-Fold(StratifiedGroupKFold)로 만든다. 같은 와인이 학습과 예측 양쪽에 들어가면 기저 모델이 정답을 외운 확률로 메타 모델을 학습하게 되어, 실제보다 기저 모델을 과신하는 결합이 만들어지기 때문이다.

**결과**
- 세 확률을 같은 비중으로 평균한 모델이 리더보드 0.709로 기존 최고 점수였고, 그룹 교차검증 정확도는 0.5827이다.
- 처음 보는 697행 정확도로 환산하면 (709 − 303) / 697 = 0.5825로 교차검증 예상치와 같아, 0.709는 운이 아니라 모델 실력만큼 나온 점수였다.

In [2]:
rf = RandomForestClassifier(n_estimators=500, max_depth=20, random_state=42)
et = ExtraTreesClassifier(n_estimators=500, max_depth=25, random_state=42)
cat = CatBoostClassifier(iterations=500, depth=8, learning_rate=0.05, loss_function='MultiClass', verbose=0, random_state=42)

_, groups = get_group_cv(train)
inner_cv = StratifiedGroupKFold(3, shuffle=True, random_state=42)
oof_proba = [cross_val_predict(clone(m), X_common, y, cv=inner_cv, groups=groups, method='predict_proba') for m in [rf, et, cat]]
common_fitted = [clone(m).fit(X_common, y) for m in [rf, et, cat]]
meta = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=2000))
meta.fit(np.hstack(oof_proba), y)
stack_proba = meta.predict_proba(np.hstack([m.predict_proba(X_test_common) for m in common_fitted]))

fitted = [clone(m).fit(X, y) for m in [rf, et]]
proba_709 = (fitted[0].predict_proba(X_test) + fitted[1].predict_proba(X_test) + stack_proba) / 3

## 2-3. TabICL을 더하는 이유

**0.709 이후 막혔던 원인**
- 0.709 이후 하이퍼파라미터, 파생 변수, 회귀 기반 모델, 부스팅, 스태킹 변형, 신경망 등 약 100가지를 시험했고, 교차검증에서 기준을 넘은 15개 파일을 제출했으나 모두 0.697~0.708이었다.
- 원인을 분석한 결과, 성분이 가장 가까운 다른 와인과 등급이 같은 비율이 41~67%에 그칠 만큼 라벨 자체의 변동이 컸고, 트리 계열 모델끼리는 0.709와 다르게 예측하는 행이 비슷해 섞어도 새로운 정보가 거의 더해지지 않았다.

**그래서 TabICL**
- TabICL은 대량의 합성 표 데이터로 미리 학습된 기초모델로, 새 데이터에서 가중치를 다시 학습하지 않고 학습 데이터 전체를 문맥으로 넣어 예측한다. 트리 모델과 학습 원리가 완전히 다르다.
- 그 결과 TabICL은 단독 정확도가 0.709 모델과 거의 같으면서도(그룹 교차검증 차이 −0.0005), 0.709와 예측이 다른 행의 비율이 약 10.7%로 트리 계열(약 7%)보다 많았다. 정확도는 비슷한데 틀리는 행이 다르다는 뜻이므로, 섞으면 서로의 오답을 보완할 수 있다.
- 입력은 원래 성분 11개와 종류(화이트=1)이고, 결과의 흔들림을 줄이기 위해 random_state 42, 7, 2026으로 세 번 학습해 확률을 평균한다.
- `device='cpu'`로 고정한다. GPU에서는 반정밀도 연산으로 확률이 미세하게 달라질 수 있기 때문이다.

In [3]:
features = [c for c in test.columns if c not in ('index', 'type')]
X_raw = train[features].assign(is_white=(train['type'] == 'white').astype(int))
X_test_raw = test[features].assign(is_white=(test['type'] == 'white').astype(int))

tabicl_proba = sum(TabICLClassifier(n_estimators=8, random_state=seed, device='cpu').fit(X_raw, y).predict_proba(X_test_raw) for seed in [42, 7, 2026]) / 3

## 2-4. 최종 결합과 제출 파일

**왜 0.25 비중인가**
- 0.709 확률 평균에 TabICL 확률을 0.25 비중으로 더하면, 0.709 모델이 확신하는 행은 그대로 두고 두 모델의 확률 차이가 작은 경계 행에서만 예측이 바뀐다.
- 그룹 교차검증(분할 3가지 × 5-Fold)에서 0.709 대비 +0.0042였고 분할 3가지 모두 상승했으며(+0.0015 / +0.0057 / +0.0054), random_state와 분할을 바꾼 재검증에서도 +0.0042로 같았다.
- 교차검증에서 예측이 바뀐 행만 보면 결합 모델의 정답률은 44~53%, 0.709 모델은 32~39%였다.

**왜 303행을 학습 데이터 품질로 맞추는가**
- EDA에서 성분이 같으면 품질이 항상 같았으므로(충돌 0건), 학습 데이터와 성분이 같은 테스트 303행은 학습 데이터의 품질이 가장 정확한 예측이다.

**결과**
- 0.709 파일과 다른 행은 16행(6→7 5행, 6→5 5행, 5→6 3행, 7→6 3행)이고, 리더보드 0.711로 이 중 2행을 더 맞혔다.
- 아래 출력은 Apple 칩 맥북(Python 3.14, requirements.txt의 버전)에서 실행한 결과이며, 품질별 개수 {4: 7, 5: 354, 6: 503, 7: 125, 8: 11}은 제출 파일과 같다.

In [4]:
final_proba = proba_709 + 0.25 * tabicl_proba
pred = fitted[0].classes_[final_proba.argmax(axis=1)]

known = make_x(train).assign(known_quality=train['quality']).drop_duplicates()
merged = make_x(test).merge(known, how='left')
pred = np.where(merged['known_quality'].notna(), merged['known_quality'], pred).astype(int)

submission = make_submission(pred, sub, 'submission_opt_709_tabicl_w0.25.csv')
pd.Series(pred).value_counts().sort_index().to_dict()

[제출 파일 점검] 이상 없음 -> submission_opt_709_tabicl_w0.25.csv 저장
quality
4      7
5    354
6    503
7    125
8     11


{4: 7, 5: 354, 6: 503, 7: 125, 8: 11}

## 2-5. 한계와 재현 조건

- 리더보드 차이 0.002는 1,000행 중 2행이므로 리더보드만으로 우열을 단정하기 어렵다. 다만 교차검증과 리더보드가 같은 방향을 가리킨 것은 0.709 이후 처음이다.
- 3·4·8·9등급은 여전히 거의 맞히지 못한다. 성분만으로는 극단 등급을 구별할 정보가 부족하다.
- 재현 조건: Intel 계열 CPU(예: Google Colab)에서는 랜덤 포레스트·Extra Trees·CatBoost의 학습 계산이 미세하게 달라 경계의 몇 행이 바뀐다(Colab 실행 결과 0.708). 제출 파일과 똑같은 결과는 Apple 칩 맥북에서 `requirements.txt`의 버전으로 실행해야 나온다.